# Examples

This jupyter notebook contains examples of running the various functions available in _archeryutils_.
Where different options exist for the examples they are listed. Users are encouraged to re-run various cells of this notebook with different options to explore the full functionality of the code.

## 0. Getting set up

To start off using archeryutils we need to import it.
Assuming it has been installed into the local environment according to the repository documentation this can be done as follows:

In [17]:
import archeryutils as au

## 1. Basic Building Blocks
The basic building blocks of the archeryutils package are the `Target` and the `Round` classes.

### Target

A target is defined with the following attributes:
- Scoring system
- Face diameter [default cm]
- Distance [default m]

The following scoring systems are possible:
- `"5_zone"`,
- `"10_zone"`,
- `"10_zone_compound"`,
- `"10_zone_6_ring"`,
- `"10_zone_5_ring"`,
- `"10_zone_5_ring_compound"`,
- `"WA_field"`,
- `"IFAA_field"`,
- `"IFAA_field_expert"`,
- `"Beiter_hit_miss"`,
- `"Worcester"`,
- `"Worcester_2_ring"`

where an `"_n_ring"` suffix indicates a reduced scoring area with only the central n rings, and the `"_compound"` suffix indicates the system where only the x-ring scores 10 points.

So we could define the target shot on the WA 720 70m round as:

In [18]:
my720target = au.Target("10_zone", 122, 70.0)

and the corresponding target shot by compounds (reduced 6-ring 80cm face at 50m) as:

In [19]:
mycompound720target = au.Target("10_zone_6_ring", 80, 50.0)

The diameter and distance can alternatively be provided as a tuple, where the first element is the magnitude and the second is a string specifying the units of measurement.
This provides the ability to specify an imperial distance in yards and face diameter in inches etc.

The target object can also take the optional boolean argument of `indoor` (default = `False`)
to indicate if the round is to be shot indoors (where rules may be different).

For example, a Worcester target can be defined as:

In [20]:
myWorcesterTarget = au.Target(
    "Worcester", diameter=(16, "inches"), distance=(20.0, "yards"), indoor=True
)

and the longest target on an IFAA field round could be defined as follows:

In [21]:
myIFAATarget = au.Target("IFAA_field", diameter=80, distance=(80.0, "yards"))

Sometimes you might want to represent a target that isn't represented by the built in scoring systems.
In this case you can manually supply the target ring sizes and scores as a `FaceSpec`, which a mapping (commonly a dict) of ring diameters to scores
and construct a target as so:

In [22]:
# Kings of archery recurve scoring target
face_spec = {8: 10, 12: 8, 16: 7, 20: 6}
myKingsTarget = au.Target.from_face_spec((face_spec, "cm"), 40, 18, indoor=True)
print(myKingsTarget.scoring_system)

Custom


Under the hood, all standard scoring systems autogenerate their own `FaceSpec` and this is used internally when calculating handicaps and classifications. You can see this stored under the `Target.face_spec` property:

In [23]:
print(my720target.face_spec)

{0.122: 10, 0.244: 9, 0.366: 8, 0.488: 7, 0.61: 6, 0.732: 5, 0.854: 4, 0.976: 3, 1.098: 2, 1.22: 1}


Target objects have the ability to return the maximum possible score from the type of face specified through the `max_score` method:

In [24]:
for target in [my720target, mycompound720target, myIFAATarget, myWorcesterTarget]:
    print(target.max_score())

10
10
5
5


### Pass
The natural extension to the `targets.Target` class is to shoot a number of arrows at it.
in _archeryutils_ this is called a "pass" and is defined using the `rounds.Pass` class which wraps around the `targets.Target` class.

This takes a number of arrows followed by all of the arguments to target defined above.

For example, to define the 36 arrow pass that forms the first distance on a WA 1440 70m round, or the first half of the WA 720 70m round we use:

In [25]:
my70mPass = au.Pass(36, my720target)

We can also bypass the Target class and directly construct our Pass using the `at_target` constructor

In [26]:
my70mPass = au.Pass.at_target(36, "10_zone", 122, 70.0)

Like the `targets.Target` class a `rounds.Pass` also has a `max_score()` method, but this now returns the maximum possible score for the pass (`n_arrows * Target.max_score()`):

In [27]:
print(my70mPass.max_score())

360


### Round
In reality we rarely use the `targets.Target` or `rounds.Pass` objects by themselves, however, instead preferring to use the `rounds.Round` class. This defines multiple passes to form what is commonly known as a round.

A `rounds.Round` object is defined with a string `name` to provide a popular name for the round and an iterable of `rounds.Pass` objects, which will be stored as a list in the `passes` attribute.

It may also take the following optional string arguments:
- `location` - where the round is shot, e.g. 'Indoor', 'Outdoor', 'Field' etc.
- `body` - The governing body the round is defined by, e.g. 'WA', 'IFAA', 'AGB', 'AA' etc.
- `family` - The larger family of rounds to which this round belongs, e.g. 'wa_1440', 'wa_720', 'national' etc.

So to define a WA 720 70m round we can re-use our variable `my70mPass` from above as follows:

In [28]:
my720Round = au.Round(
    "WA 720 (70m)",
    [my70mPass, my70mPass],
    location="Outdoor Target",
    body="WA",
    family="WA720",
)

Again we have a method for maximum score:

In [29]:
print(my720Round.max_score())

720


### Default Rounds

A number of useful rounds are pre-defined and come preloaded as dictionaries that can be imported:

In [30]:
from archeryutils import load_rounds

agb_outdoor_imperial = load_rounds.AGB_outdoor_imperial

for round_i in agb_outdoor_imperial.values():
    print(round_i.name)

York
Hereford
Bristol I
Bristol II
Bristol III
Bristol IV
Bristol V
St. George
Albion
Windsor
Windsor 50
Windsor 40
Windsor 30
New Western
Long Western
Western
Western 50
Western 40
Western 30
American
St Nicholas
New National
Long National
National
National 50
National 40
National 30
New Warwick
Long Warwick
Warwick
Warwick 50
Warwick 40
Warwick 30


The individial rounds are accessible via 'dot' notation (using the alias listed in `agb_outdoor_imperial.keys()`) as follows:

In [31]:
agb_outdoor_imperial.york.get_info()

agb_outdoor_imperial.york.max_score()

A York consists of 3 passes:
	- 72 arrows at a 122.0 cm target at 100.0 yards.
	- 48 arrows at a 122.0 cm target at 80.0 yards.
	- 24 arrows at a 122.0 cm target at 60.0 yards.


1296

Possible options for round collections are:
- `AGB_outdoor_imperial` - Archery GB outdoor imperial rounds
- `AGB_outdoor_metric` - Archery GB outdoor metric rounds
- `AGB_indoor` - Archery GB indoor rounds
- `WA_outdoor` - World Archery outdoor rounds
- `WA_indoor` - World Archery indoor rounds
- `WA_field` - World Archery field rounds
- `IFAA_field` - IFAA indoor and outdoor rounds
- `AGB_VI` - Archery GB Visually Impaired rounds
- `WA_VI` - World Archery Visually Impaired rounds
- `custom` - custom rounds such as individual distances, 252 awards, frostbites etc.

## 2. Handicap Schemes
_archeryutils_ provides functionality for calculating various [handicaps/skill ratings](https://jackatkinson.net/post/archery_handicap/) from scores. These include both the popular Archery GB and Archery Australia schemes.

To use these functionalities import the handicaps module as below.

In [32]:
from archeryutils import handicaps as hc

### Score from handicap

It is then possible to use the `score_for_round` function to calculate score on any `rounds.Round` for a given handicap/skill rating.

This requires a round, handicap/skill rating, scheme, and set of handicap parameters.

Possible options for the scheme are:
- `"AGB"` - The 2023 Archery GB handicap system developed by Jack Atkinson
- `"AGBold"` - The old Archery GB handicap system developed by David Lane
- `"AA2"` - The 2014 Archery Australia Skill rating system developed by Jim Park
- `"AA"` - The old Archery Australia skill rating system developed by Jim Park

For example, to calculate the score on a York round for a handicap of 38 using the 2023 Archery GB scheme we run:

In [33]:
score_from_hc = hc.score_for_round(
    38,
    agb_outdoor_imperial.york,
    "AGB",
)

print(f"A handicap of 38 on a York is a score of {score_from_hc}.")

A handicap of 38 on a York is a score of 940.0.


Note that it is possible to obtain scores for decimal handicaps:

In [34]:
score_from_hc = hc.score_for_round(
    38.25,
    agb_outdoor_imperial.york,
    "AGB",
)

print(f"A handicap of 38.25 on a York is a score of {score_from_hc}.")

A handicap of 38.25 on a York is a score of 936.0.


By default this function returns a round score as would appear in handicap tables and is physically attainable when shooting a round. The rounding mechanism (round/floor/ceil) varies by scheme. However, it is possible to return the mathematically continuous score by setting the `rounded_score` optional argument to be `False`:

In [35]:
score_from_hc = hc.score_for_round(
    38.25,
    agb_outdoor_imperial.york,
    "AGB",
    rounded_score=False,
)

print(f"A handicap of 38.25 on a York is a decimal score of {score_from_hc}.")

A handicap of 38.25 on a York is a decimal score of 935.1392717845215.


It is also possible to get the predicted score for each pass in a round using the `score_for_passes` function:

In [36]:
pass_scores = hc.score_for_passes(
    38,
    agb_outdoor_imperial.york,
    "AGB",
)

print(f"A handicap of 38 on a York gives pass scores of {pass_scores}.")

A handicap of 38 on a York gives pass scores of [421. 331. 188.].


### Handicap from Score

Mathematically is is easy to define a score for a given handicap, but often the opposite is required, where one wishes to obtain the handicap given a score.

To perform this operation use the `handicap_from_score()` function which takes a score, round, and handicap scheme.
By default it returns the decimal handicap corresponding to the provided score exactly.
However, it is possible to return the integer handicap value that the score would correspond to in a handicap table by setting the `int_prec` optional argument to `True`. Remember that the rounding mechanism (round/floor/ceil) varies by scheme.


For example, to get the 2023 Archery GB handicap given by a score of 950 on a York round:

In [37]:
hc_from_score = hc.handicap_from_score(
    950,
    agb_outdoor_imperial.york,
    "AGB",
)
print(f"A score of 950 on a York is a continuous handicap of {hc_from_score}.")

hc_from_score = hc.handicap_from_score(
    950,
    agb_outdoor_imperial.york,
    "AGB",
    int_prec=True,
)
print(f"A score of 950 on a York is a discrete handicap of {hc_from_score}.")

A score of 950 on a York is a continuous handicap of 37.374134403932665.
A score of 950 on a York is a discrete handicap of 38.0.


### Handicap Tables

A further functionality of the code is the ability to generate "Handicap Tables" for arbitrary lists of rounds and handicaps.

To do this use the `HandicapTable` object which can be initialised with the name of a handicap scheme, an array of handicaps to display scores for, and a list of `Round`s to display scores for.
The `HandicapTable` object stores the table, but it can be printed to the output using `print()`:

In [38]:
import numpy as np

handicaps = np.arange(0.0, 151.0, 1.0)
rounds = [
    agb_outdoor_imperial.york,
    agb_outdoor_imperial.hereford,
    agb_outdoor_imperial.albion,
    agb_outdoor_imperial.windsor,
]
# The following would allow handicap tables for an entire group of rounds to be generated:
# rounds = list(load_rounds.AGB_outdoor_imperial.values())

agb_handicap_table = hc.HandicapTable(
    "AGB",
    handicaps,
    rounds,
)

print(agb_handicap_table)

      Handicap          York      Hereford        Albion       Windsor
             0          1284          1295           972              
             1          1281                                          
             2          1279          1294                            
             3          1276          1293           971              
             4          1272          1292           970              
             5          1268          1290                            
             6          1264          1289           969              
             7          1260          1287           968              
             8          1255          1285           967              
             9          1250          1283           966              
            10          1245          1281           964           972
            11          1239          1278           963              
            12          1233          1275           961              
      

The following optional arguments can be passed to `HandicapTable`:
- `rounded_scores` - if `False`, display decimal scores instead of rounding to discrete values as appropriate for the scheme
- `clean_gaps` - if `True`, duplicate scores will be displayed for the first occurrence only
- `int_prec` - if `True` then values will be printed as integers rather than decimals

It is also possible to pass an array of non-integer handicaps.

The effect of these variables can be examined by changing their values in the following:

In [39]:
handicaps = np.arange(0.0, 51.0, 0.5)

agb_decimal_table = hc.HandicapTable(
    "AGB",
    handicaps,
    rounds,
    rounded_scores=False,
    clean_gaps=False,
    int_prec=False,
)

print(agb_decimal_table)

      Handicap          York      Hereford        Albion       Windsor
           0.0 1283.43539112 1294.41128340  971.20310600  971.99244624
           0.5 1282.23101439 1294.14732559  971.07019646  971.98968696
           1.0 1280.95163666 1293.85033513  970.92047684  971.98606417
           1.5 1279.59589344 1293.51777056  970.75259933  971.98135501
           2.0 1278.16257099 1293.14707366  970.56519185  971.97529244
           2.5 1276.65060131 1292.73568446  970.35686337  971.96755961
           3.0 1275.05905507 1292.28105576  970.12620874  971.95778415
           3.5 1273.38713286 1291.78066714  969.87181326  971.94553236
           4.0 1271.63415490 1291.23203803  969.59225663  971.93030358
           4.5 1269.79954959 1290.63273977  969.28611639  971.91152478
           5.0 1267.88284129 1289.98040647  968.95197074  971.88854538
           5.5 1265.88363751 1289.27274454  968.58840071  971.86063266
           6.0 1263.80161600 1288.50754076  968.19399174  971.82696765
      

The `HandicapTable` class also contains two methods for saving the table to file:
- `to_file(<filename>)` - which will save the nicely formatted table as seen from `print()` as an ascii file with the provided filename
- `to_csv(<filename>)` - saves the table in a comma-separated-variable format


In [40]:
agb_handicap_table.to_file("my_saved_table.txt")

agb_handicap_table.to_csv("my_saved_csv_table.csv")

Writing handicap table to file...Done.
Writing handicap table to csv...Done.


## 3. Classifications

As well as handicap functionalities _archeryutils_ fontains functionalities for calculating Archery GB classifications.
These are accessed by importing the `classifications` module:

In [41]:
from archeryutils import classifications as cf

### Category Enums

The category for which we request a classification is made up of three components under the Archery GB systems:

* Age
* Bowstyle
* Gender

To represent these `archeryutils` utilises three enums.
They can be imported from `archeryutils.classifications` and are demonstrated in the following cell.
Specific values can be accessed using dot notation as we will see in subsequent examples.

In [42]:
from archeryutils.classifications import AGB_ages, AGB_bowstyles, AGB_genders

print("AGB_ages:")
for age in AGB_ages:
    print(f"\t{age.name}")

print("AGB_bowstyles:")
for bowstyle in AGB_bowstyles:
    print(f"\t{bowstyle.name}")

print("AGB_genders:")
for gender in AGB_genders:
    print(f"\t{gender.name}")

AGB_ages:
	OVER_50
	ADULT
	UNDER_21
	UNDER_18
	UNDER_16
	UNDER_15
	UNDER_14
	UNDER_12
AGB_bowstyles:
	COMPOUND
	RECURVE
	BAREBOW
	LONGBOW
	TRADITIONAL
	FLATBOW
	COMPOUNDLIMITED
	COMPOUNDBAREBOW
AGB_genders:
	OPEN
	FEMALE


### Classification from score

To get a classification that results from a score use the `calculate_<X>_classification()` function, where `X` corresponds to the classification scheme being used (`agb_outdoor`, `agb_indoor`, `agb_field`).

This takes following arguments:
- a score
- an archeryutils `Round` suitable for the particular classification scheme (or a string of a round's alias -- see [Default Rounds](#Default-Rounds).
- an `AGB_bowstyles` value
- an `AGB_genders` value
- an `AGB_ages` value

and returns a string corresponding to the classification it obtains.

These can be investigated in the following code snippet which uses a number of examples:

In [43]:
# AGB Outdoor

agb_outdoor_imperial = load_rounds.AGB_outdoor_imperial

class_from_score = cf.calculate_agb_outdoor_classification(
    965,
    agb_outdoor_imperial.hereford,
    AGB_bowstyles.RECURVE,
    AGB_genders.OPEN,
    AGB_ages.OVER_50,
)
print(
    f"A score of 965 on a Hereford is class {class_from_score} for a 50+ open recurve."
)

A score of 965 on a Hereford is class B2 for a 50+ open recurve.


In [44]:
# AGB Indoor

agb_indoor = load_rounds.AGB_indoor
wa_indoor = load_rounds.WA_indoor

class_from_score = cf.calculate_agb_indoor_classification(
    562,
    wa_indoor.wa18,
    AGB_bowstyles.COMPOUND,
    AGB_genders.FEMALE,
    AGB_ages.ADULT,
)
print(
    f"A score of 562 on a WA 18 is class {class_from_score} for adult female compound."
)

A score of 562 on a WA 18 is class I-B1 for adult female compound.


In [45]:
# AGB Field

wa_field = load_rounds.WA_field

class_from_score = cf.calculate_agb_field_classification(
    168,
    wa_field.wa_field_24_blue_unmarked,
    AGB_bowstyles.TRADITIONAL,
    AGB_genders.OPEN,
    AGB_ages.UNDER_18,
)
print(
    f"A score of 168 on a WA Unmarked 24 is class {class_from_score} for an under 18 open traditional."
)

A score of 168 on a WA Unmarked 24 is class B1 for an under 18 open traditional.


### Classification scores

As well as generating a classification from a score there is also the inverse functionality of obtaining scores required for classifications. This can be done using the `<X>_classification_scores()` functions.

These take a round alias and categories as strings above, and return a list of scores required for each classification in descending order.

Where a classification is not available from a particular round a fill value of -9999 is returned.

In [46]:
class_scores = cf.agb_outdoor_classification_scores(
    agb_outdoor_imperial.hereford,
    AGB_bowstyles.RECURVE,
    AGB_genders.OPEN,
    AGB_ages.ADULT,
)
print(class_scores)

[-9999, -9999, -9999, -9999, 985, 863, 721, 571, 427]


In [47]:
class_scores = cf.agb_indoor_classification_scores(
    agb_indoor.portsmouth,
    AGB_bowstyles.COMPOUND,
    AGB_genders.FEMALE,
    AGB_ages.ADULT,
)
print(class_scores)

[589, 577, 566, 555, 541, 521, 491, 449]


In [48]:
class_scores = cf.agb_field_classification_scores(
    wa_field.wa_field_24_blue_marked,
    AGB_bowstyles.FLATBOW,
    AGB_genders.FEMALE,
    AGB_ages.UNDER_18,
)
print(class_scores)

[164, 132, 104, 80, 60, 44, 32, 23, 16]


### Category Coaxing (advanced use)

The rules on classification schemes mean that some bowstyles/ages are not valid, for example the following code will return an error:
```python
class_scores = cf.agb_outdoor_classification_scores(
    agb_outdoor_imperial.hereford,
    AGB_bowstyles.FLATBOW,
    AGB_genders.FEMALE,
    AGB_ages.ADULT,
)
```
```console
ValueError: AGB_bowstyles.FLATBOW is not a recognised bowstyle for outdoor classifications. Please select from `AGB_bowstyles.COMPOUND|RECURVE|BAREBOW|LONGBOW`.
```
as there is no "flatbow" category for target.

If we wish to force a valid result we can use the coaxing functions to transform the input to a valid one (represented by a `dict`). In this example "flatbow" will be 'upgraded' per the Archery GB rules to barebow.

In [49]:
cf.coax_outdoor_group(
    AGB_bowstyles.FLATBOW,
    AGB_genders.FEMALE,
    AGB_ages.ADULT,
)

{'bowstyle': <AGB_bowstyles.BAREBOW: 4>,
 'gender': <AGB_genders.FEMALE: 2>,
 'age_group': <AGB_ages.ADULT: 2>}

The beauty of this implementation is that the function can be used, via dictionary unpacking, directly in calls to the other classification functions:

In [50]:
class_scores = cf.agb_outdoor_classification_scores(
    agb_outdoor_imperial.hereford,
    **cf.coax_outdoor_group(
        AGB_bowstyles.FLATBOW,
        AGB_genders.FEMALE,
        AGB_ages.ADULT,
    ),
)

print(class_scores)

[1023, 936, 834, 721, 603, 487, 380, 287, 212]
